# Export processed samples (features-faithful)

Write a small set of audio under `training/processed-samples/` for listening to **what the feature pipelines actually use**, not human-sweetened mixes.

| Condition | What is written | Relation to model input |
|-----------|-----------------|-------------------------|
| **dry** | mono 48 kHz load of source clip | Pre-CQT dry waveform |
| **rir** | exact RIR conv → crop → peak-norm (`mix-rir-cqt`) | Exact pre-CQT wet |
| **dir** | exact DIR conv → crop → peak-norm (`mix-dir-cqt`) | Exact pre-CQT wet |
| **noise_cqt_inv** | Griffin–Lim inverse of **stored mixed CQT** | Approx. listen of noise features |
| **clean_cqt_inv** | Griffin–Lim inverse of **stored clean CQT** | A/B for noise (feature domain) |

Noise features are power-domain CQT mixes (not `x+n`). RIR/DIR wavs are bit-faithful to pre-CQT audio.

## Config

In [1]:
from os import path
import pandas as pd
from IPython.display import display

FEATURES_DIR = path.normpath("../features")
RIR_BANK = path.join(FEATURES_DIR, "rir/ir-survey-indoor_no_bathroom.npz")
DIR_BANK = path.join(FEATURES_DIR, "dir/dirs-mic6.npz")
OUT_ROOT = path.normpath("../processed-samples")

DOMAINS = ["vivo", "thinkpad", "flow"]
DATASET_ROOTS = {
    "vivo": path.normpath("../datasets/vivo"),
    "thinkpad": path.normpath("../datasets/thinkpad"),
    "flow": path.normpath("../evaluations/test-all"),
}

N_PER_DOMAIN = 4
SEED = 42
INCLUDE_EXTREME_RIR = True
GL_N_ITER = 32

AUDIO_SAMPLE_RATE = 48_000
CQT_HOP_LENGTH = 512
CQT_OCTAVES = 6
CQT_BINS_PER_OCTAVE = 36
CQT_N_BINS = CQT_BINS_PER_OCTAVE * CQT_OCTAVES

FORCE_REEXPORT = True  # refresh dataset including new DIR wavs

config_df = pd.DataFrame([
    {"key": "out_root", "value": OUT_ROOT},
    {"key": "domains", "value": DOMAINS},
    {"key": "n_per_domain", "value": N_PER_DOMAIN},
    {"key": "seed", "value": SEED},
    {"key": "rir_bank", "value": RIR_BANK},
    {"key": "dir_bank", "value": DIR_BANK},
    {"key": "gl_n_iter", "value": GL_N_ITER},
    {"key": "force_reexport", "value": FORCE_REEXPORT},
]).set_index("key")
display(config_df)

,value
key,
out_root,../processed-samples
domains,"[vivo, thinkpad, flow]"
n_per_domain,4
seed,42
rir_bank,../features/rir/ir-survey-indoor_no_bathroom.npz
dir_bank,../features/dir/dirs-mic6.npz
gl_n_iter,32
force_reexport,True


## Helpers (same processing as pipelines)

In [2]:
from os import makedirs
from pathlib import Path
import re

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import fftconvolve
from IPython.display import display
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

CQT_FMIN = librosa.note_to_hz("C1")


def safe_stem(s: str) -> str:
    s = s.replace("/", "__").replace("\\", "__")
    return re.sub(r"[^A-Za-z0-9._#+-]+", "_", s)


def convolve_crop(y: np.ndarray, h: np.ndarray) -> np.ndarray:
    """Identical to mix-rir-cqt / mix-dir-cqt."""
    wet = fftconvolve(y, h, mode="full")
    wet = wet[: len(y)]
    peak = float(np.max(np.abs(wet)))
    if peak > 0:
        wet = wet / peak
    return wet.astype(np.float32)


def load_mono(path_str: str) -> np.ndarray:
    y, _ = librosa.load(path_str, sr=AUDIO_SAMPLE_RATE, mono=True)
    return y.astype(np.float32)


def write_wav(path_str: str, y: np.ndarray):
    y = np.asarray(y, dtype=np.float32)
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    sf.write(path_str, y, AUDIO_SAMPLE_RATE, subtype="FLOAT")
    return {"path": path_str, "n_samples": int(len(y)), "peak": peak}


def cqt_db_to_audio(cqt_db: np.ndarray) -> np.ndarray:
    mag = librosa.db_to_amplitude(cqt_db.astype(np.float64), ref=1.0)
    y = librosa.griffinlim_cqt(
        mag,
        sr=AUDIO_SAMPLE_RATE,
        hop_length=CQT_HOP_LENGTH,
        fmin=CQT_FMIN,
        bins_per_octave=CQT_BINS_PER_OCTAVE,
        n_iter=GL_N_ITER,
    )
    return y.astype(np.float32)


def resolve_audio_path(domain: str, relpath: str) -> str:
    root = Path(DATASET_ROOTS[domain])
    p = root / relpath
    if p.is_file():
        return str(p)
    for alt_suf in (".ogg", ".wav", ".flac"):
        q = p.with_suffix(alt_suf)
        if q.is_file():
            return str(q)
    raise FileNotFoundError(f"{domain}: missing {p}")


def load_rir_bank():
    data = np.load(RIR_BANK, allow_pickle=True)
    return {
        "irs": data["irs"],
        "names": data["names"].astype(str),
        "rooms": data["rooms"].astype(str),
    }


def load_dir_bank():
    data = np.load(DIR_BANK, allow_pickle=True)
    devices = data["devices"].astype(str) if "devices" in data.files else data["rooms"].astype(str)
    return {
        "irs": data["irs"],
        "names": data["names"].astype(str),
        "devices": devices,
    }


def pick_indices(n_total: int, n_pick: int, rng: np.random.Generator) -> np.ndarray:
    n_pick = min(n_pick, n_total)
    return np.sort(rng.choice(n_total, size=n_pick, replace=False))


display(pd.DataFrame([{"helpers": "ready"}]))

,helpers
0,ready


## Select clips + export

In [3]:
from tqdm.notebook import tqdm

assert path.isfile(RIR_BANK), f"Missing RIR bank: {RIR_BANK}"
assert path.isfile(DIR_BANK), f"Missing DIR bank: {DIR_BANK} (run dir-prep)"
rir_bank = load_rir_bank()
dir_bank = load_dir_bank()
rir_name_to_idx = {n: i for i, n in enumerate(rir_bank["names"])}
dir_name_to_idx = {n: i for i, n in enumerate(dir_bank["names"])}

makedirs(OUT_ROOT, exist_ok=True)
index_rows = []

for d_i, domain in enumerate(DOMAINS):
    rir_man_path = path.join(FEATURES_DIR, f"{domain}-rir-indoor_no_bathroom.manifest.csv")
    dir_man_path = path.join(FEATURES_DIR, f"{domain}-dir-mic6.manifest.csv")
    noise_man_path = path.join(FEATURES_DIR, f"{domain}-noise-interior_domestic.manifest.csv")
    clean_npz = path.join(FEATURES_DIR, f"{domain}.npz")
    noise_npz = path.join(FEATURES_DIR, f"{domain}-noise-interior_domestic.npz")

    for p, label in [
        (rir_man_path, "rir_manifest"),
        (dir_man_path, "dir_manifest"),
        (noise_man_path, "noise_manifest"),
        (clean_npz, "clean_features"),
        (noise_npz, "noise_features"),
    ]:
        assert path.isfile(p), f"Missing {label}: {p}"

    rir_man = pd.read_csv(rir_man_path)
    dir_man = pd.read_csv(dir_man_path)
    noise_man = pd.read_csv(noise_man_path)
    clean = np.load(clean_npz, allow_pickle=True)
    noise = np.load(noise_npz, allow_pickle=True)

    n = int(clean["features"].shape[0])
    assert len(rir_man) == n and len(dir_man) == n and len(noise_man) == n
    assert noise["features"].shape[0] == n

    rng = np.random.default_rng(SEED + 100 + d_i)
    idxs = list(pick_indices(n, N_PER_DOMAIN, rng))

    if INCLUDE_EXTREME_RIR:
        room_dur = {}
        for name, room in zip(rir_bank["names"], rir_bank["rooms"]):
            room_dur.setdefault(room, []).append(len(rir_bank["irs"][rir_name_to_idx[name]]))
        room_med = {r: float(np.median(v)) for r, v in room_dur.items()}
        tmp = rir_man.copy()
        tmp["_room_len"] = tmp["rir_room"].map(lambda r: room_med.get(r, 0.0))
        extreme_i = int(tmp["_room_len"].idxmax())
        if extreme_i not in idxs:
            idxs.append(extreme_i)

    out_dir = path.join(OUT_ROOT, domain)
    makedirs(out_dir, exist_ok=True)

    for i in tqdm(idxs, desc=f"export {domain}"):
        row_rir = rir_man.iloc[i]
        row_dir = dir_man.iloc[i]
        row_noise = noise_man.iloc[i]
        chord_file = str(row_rir["chord_file"])
        label = str(row_rir["chord_label"])
        stem = safe_stem(f"{i:04d}__{chord_file}")

        audio_path = resolve_audio_path(domain, chord_file)
        y_dry = load_mono(audio_path)

        # RIR (manifest IR)
        rir_name = str(row_rir["rir_file"])
        rir_i = int(row_rir["rir_index"])
        if 0 <= rir_i < len(rir_bank["irs"]):
            h_rir = rir_bank["irs"][rir_i]
            if rir_bank["names"][rir_i] != rir_name and rir_name in rir_name_to_idx:
                h_rir = rir_bank["irs"][rir_name_to_idx[rir_name]]
        else:
            h_rir = rir_bank["irs"][rir_name_to_idx[rir_name]]
        y_rir = convolve_crop(y_dry, h_rir)

        # DIR (manifest DIR)
        dir_name = str(row_dir["dir_file"])
        dir_i = int(row_dir["dir_index"])
        if 0 <= dir_i < len(dir_bank["irs"]):
            h_dir = dir_bank["irs"][dir_i]
            if dir_bank["names"][dir_i] != dir_name and dir_name in dir_name_to_idx:
                h_dir = dir_bank["irs"][dir_name_to_idx[dir_name]]
        else:
            h_dir = dir_bank["irs"][dir_name_to_idx[dir_name]]
        y_dir = convolve_crop(y_dry, h_dir)

        dry_path = path.join(out_dir, f"{stem}__dry.wav")
        rir_path = path.join(out_dir, f"{stem}__rir.wav")
        dir_path = path.join(out_dir, f"{stem}__dir.wav")
        if FORCE_REEXPORT or not path.isfile(dry_path):
            write_wav(dry_path, y_dry)
        if FORCE_REEXPORT or not path.isfile(rir_path):
            write_wav(rir_path, y_rir)
        if FORCE_REEXPORT or not path.isfile(dir_path):
            write_wav(dir_path, y_dir)

        clean_cqt = clean["features"][i]
        noise_cqt = noise["features"][i]
        assert clean_cqt.shape[0] == CQT_N_BINS

        clean_inv_path = path.join(out_dir, f"{stem}__clean_cqt_inv.wav")
        noise_inv_path = path.join(out_dir, f"{stem}__noise_cqt_inv.wav")
        if FORCE_REEXPORT or not path.isfile(clean_inv_path):
            write_wav(clean_inv_path, cqt_db_to_audio(clean_cqt))
        if FORCE_REEXPORT or not path.isfile(noise_inv_path):
            write_wav(noise_inv_path, cqt_db_to_audio(noise_cqt))

        index_rows.append({
            "domain": domain,
            "index": int(i),
            "chord_file": chord_file,
            "chord_label": label,
            "source_audio": audio_path,
            "rir_file": rir_name,
            "rir_room": str(row_rir["rir_room"]),
            "rir_index": int(row_rir["rir_index"]),
            "dir_file": dir_name,
            "dir_device": str(row_dir["dir_device"]),
            "dir_index": int(row_dir["dir_index"]),
            "noise_file": str(row_noise.get("noise_file", "")),
            "noise_category": str(row_noise.get("noise_category", "")),
            "snr_db": row_noise.get("snr_db", np.nan),
            "time_roll": row_noise.get("time_roll", np.nan),
            "dry_wav": dry_path,
            "rir_wav": rir_path,
            "dir_wav": dir_path,
            "clean_cqt_inv_wav": clean_inv_path,
            "noise_cqt_inv_wav": noise_inv_path,
        })

index_df = pd.DataFrame(index_rows)
index_path = path.join(OUT_ROOT, "index.csv")
index_df.to_csv(index_path, index=False)
display(index_df[[
    "domain", "index", "chord_label", "rir_room", "dir_device", "noise_category", "snr_db",
]])
display(pd.DataFrame([{
    "index_csv": index_path,
    "n_clips": len(index_df),
    "n_wavs_expected": len(index_df) * 5,
    "out_root": OUT_ROOT,
}]))

export vivo:   0%|          | 0/5 [00:00<?, ?it/s]

export thinkpad:   0%|          | 0/5 [00:00<?, ?it/s]

export flow:   0%|          | 0/5 [00:00<?, ?it/s]

,domain,index,chord_label,rir_room,dir_device,noise_category,snr_db
0,vivo,22,A#_diminished_4,MITCampus,STC4035,keyboard_typing,23.182243
1,vivo,32,A#_diminished_4,Classroom,AKGD12,clock_alarm,14.250389
2,vivo,154,A_diminished_4,Classroom,STC4035,mouse_click,19.785684
3,vivo,760,D_major_4,MITCampus,AKGD12,can_opening,19.459347
4,vivo,19,A#_diminished_4,Stairwell,AKGD12,clock_tick,22.019922
5,thinkpad,594,C_minor_4,ArtGallery,OktavaMD57,vacuum_cleaner,10.141951
6,thinkpad,970,F#_diminished_4,Gym,OktavaMD57,glass_breaking,22.649236
7,thinkpad,1327,G_diminished_4,Hallway,Lomo52A5M,mouse_click,19.538462
8,thinkpad,1406,G_minor_4,MITCampus,Crystal,glass_breaking,16.783704
9,thinkpad,10,A#_diminished_4,Stairwell,Crystal,keyboard_typing,12.198393


,index_csv,n_clips,n_wavs_expected,out_root
0,../processed-samples/index.csv,15,75,../processed-samples


## Verify

In [4]:
from pathlib import Path

rows = []
for _, r in index_df.iterrows():
    for key in ["dry_wav", "rir_wav", "dir_wav", "clean_cqt_inv_wav", "noise_cqt_inv_wav"]:
        p = r[key]
        info = sf.info(p)
        rows.append({
            "domain": r["domain"],
            "index": r["index"],
            "kind": key.replace("_wav", ""),
            "exists": path.isfile(p),
            "sr": info.samplerate,
            "duration_s": round(info.duration, 3),
            "subtype": info.subtype,
            "bytes": Path(p).stat().st_size,
        })

verify_df = pd.DataFrame(rows)
display(verify_df.groupby(["domain", "kind"]).size().rename("n").reset_index())
display(verify_df.head(15))
assert verify_df["exists"].all()
assert (verify_df["sr"] == AUDIO_SAMPLE_RATE).all()
display(pd.DataFrame([{
    "total_wavs": len(verify_df),
    "total_mb": round(verify_df["bytes"].sum() / 1e6, 2),
    "index_csv": path.join(OUT_ROOT, "index.csv"),
}]))

,domain,kind,n
0,flow,clean_cqt_inv,5
1,flow,dir,5
2,flow,dry,5
3,flow,noise_cqt_inv,5
4,flow,rir,5
5,thinkpad,clean_cqt_inv,5
6,thinkpad,dir,5
7,thinkpad,dry,5
8,thinkpad,noise_cqt_inv,5
9,thinkpad,rir,5


,domain,index,kind,exists,sr,duration_s,subtype,bytes
0,vivo,22,dry,True,48000,2.640,FLOAT,506960
1,vivo,22,rir,True,48000,2.640,FLOAT,506960
2,vivo,22,dir,True,48000,2.640,FLOAT,506960
3,vivo,22,clean_cqt_inv,True,48000,1.995,FLOAT,383056
4,vivo,22,noise_cqt_inv,True,48000,1.995,FLOAT,383056
5,vivo,32,dry,True,48000,2.280,FLOAT,437840
6,vivo,32,rir,True,48000,2.280,FLOAT,437840
7,vivo,32,dir,True,48000,2.280,FLOAT,437840
8,vivo,32,clean_cqt_inv,True,48000,1.995,FLOAT,383056
9,vivo,32,noise_cqt_inv,True,48000,1.995,FLOAT,383056


,total_wavs,total_mb,index_csv
0,75,32.3,../processed-samples/index.csv
